Week 5 — RAG Expert Knowledge Worker

In [ ]:
# imports
import os
import glob
from dotenv import load_dotenv
import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Environment Setup
# Load environment variables
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"OpenAI API key loaded (starts with {api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")

In [ ]:
# Model Configuration
MODEL = "gpt-4.1-mini"
DB_NAME = "week5_rag_db"

In [ ]:
# Knowledge Base Path
ROOT = os.path.abspath(os.getcwd())

if "community-contributions" in ROOT:
    KB_PATH = os.path.normpath(os.path.join(ROOT, "..", "..", "knowledge-base"))
else:
    KB_PATH = os.path.normpath(os.path.join(ROOT, "knowledge-base"))

print(f"Knowledge base path: {KB_PATH}")
print(f"Exists: {os.path.isdir(KB_PATH)}")

In [ ]:
# Load Documents
documents = []

for folder in sorted(glob.glob(os.path.join(KB_PATH, "*"))):

    if not os.path.isdir(folder):
        continue

    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )

    docs = loader.load()

    for doc in docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
# Split Documents into Chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")

In [ ]:
# Create Embeddings + Vector Database
embeddings = OpenAIEmbeddings()

if os.path.exists(DB_NAME):
    Chroma(
        persist_directory=DB_NAME,
        embedding_function=embeddings
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
)

print(f"Vector store ready with {vectorstore._collection.count()} chunks")

In [ ]:
# Setup LLM + Retriever
llm = ChatOpenAI(
    model=MODEL,
    temperature=0
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)

In [ ]:
# RAG Prompt
RAG_PROMPT = """
You are an expert assistant for Insurellm, an insurance tech company.

Answer the question using ONLY the context provided.

If the answer is not in the context, say you cannot find it.

Be accurate and concise.

Context:
{context}

Question: {question}

Answer:
"""

In [ ]:
# RAG Functions
def rag_invoke(question):

    docs = retriever.invoke(question)

    context = format_docs(docs)

    prompt = ChatPromptTemplate.from_messages(
        [("human", RAG_PROMPT)]
    )

    chain = prompt | llm | StrOutputParser()

    answer = chain.invoke({
        "context": context,
        "question": question
    })

    # Extract document sources
    sources = []
    for d in docs:
        source = d.metadata.get("source", "unknown")
        doc_type = d.metadata.get("doc_type", "document")
        sources.append(f"{doc_type}: {os.path.basename(source)}")

    sources = list(set(sources))

    return answer, sources

In [ ]:
# Chat Function
def chat(message, history):

    if not message or not message.strip():
        return "Ask a question about Insurellm."

    try:
        return rag_invoke(message.strip())

    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Gradio Chat Interface
gr.ChatInterface(
    chat,
    title="Week 5 — RAG Expert Knowledge Worker",
    description="Ask questions about Insurellm employees, products, contracts, or company information.",
    type="messages",
    examples=[
        "Who is Emily Carter?",
        "What is Claimllm?",
        "Which products does Insurellm offer?"
    ],
).launch()
